# The Complete Guide on Neural Networks using Tensorflow

Welcome to Deep Learning with TensorFlow. While standard Machine Learning algorithms (like Logistic Regression) draw straight lines to separate data, Neural Networks can draw complex, highly non-linear boundaries.

They do this by passing data through layers of **Neurons**. Each neuron performs a simple linear operation:
$$z = X \cdot W + b$$
But the magic happens when we pass $z$ through an **Activation Function** (like ReLU or Sigmoid), which mathematically bends the output, allowing the network to learn curves and complex shapes.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing TensorFlow and data processing libraries.
2. **Data Generation:** Creating a non-linear "Circles" dataset.
3. **Architecture Design:** Building a Multi-Layer Perceptron (MLP) using the Keras Sequential API.
4. **Compilation:** Defining the mathematical rules for learning (Optimizer and Loss).
5. **Training & Evaluation:** Running Backpropagation and analyzing the network's accuracy.

In [ ]:
# Cell 2: Environment Setup and Imports

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# Set random seeds for analytical reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow Version: {tf.__version__}")
# Check if TensorFlow detects a GPU for hardware acceleration
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

## Step 1: Generating a Non-Linear Dataset

To demonstrate why Neural Networks are necessary, we will generate the `make_circles` dataset. This consists of a small circle of data points entirely enclosed by a larger circle of data points. 

A standard linear classifier cannot solve this because it is mathematically impossible to draw a single straight line that separates the inner circle from the outer circle. We need deep learning.

In [ ]:
# Cell 4: Generating and Visualizing the Data

# Generate 1000 data points consisting of two concentric circles
# noise adds realism, factor determines the distance between circles
X, y = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)

# Split the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Visualize the dataset
plt.figure(figsize=(7, 7))
plt.scatter(X_train[y_train==0][:, 0], X_train[y_train==0][:, 1], label="Class 0 (Outer)", alpha=0.7)
plt.scatter(X_train[y_train==1][:, 0], X_train[y_train==1][:, 1], label="Class 1 (Inner)", alpha=0.7)
plt.title("Concentric Circles Dataset", fontsize=14)
plt.xlabel("Feature X1")
plt.ylabel("Feature X2")
plt.legend()
plt.show()

print(f"Training Features Shape: {X_train.shape}") # Expect (800, 2)
print(f"Training Labels Shape: {y_train.shape}")   # Expect (800,)

## Step 2: Designing the Neural Network Architecture

TensorFlow's high-level API is called **Keras**. We will use the `Sequential` API to stack layers sequentially.

* **Hidden Layer 1:** 16 neurons. We use the **ReLU** (Rectified Linear Unit) activation function. It zeros out negative numbers and passes positive numbers through unchanged. This acts as the mathematical "hinge" that allows the network to bend its decision boundary.
* **Hidden Layer 2:** 16 neurons with ReLU to deepen the network's analytical capacity.
* **Output Layer:** 1 neuron. Because this is binary classification (Class 0 or 1), we use the **Sigmoid** activation function to mathematically squash the final output into a clean probability between 0.0 and 1.0.

In [ ]:
# Cell 6: Building the Keras Sequential Model

# Instantiate the Sequential model
model = tf.keras.Sequential([
    # Hidden Layer 1: 16 neurons, ReLU activation, expecting 2 input features
    tf.keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    
    # Hidden Layer 2: 16 neurons, ReLU activation
    tf.keras.layers.Dense(16, activation='relu'),
    
    # Output Layer: 1 neuron, Sigmoid activation for binary probability
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Analytically view the network's structure and parameter count
model.summary()

# Note on Parameters: 
# Layer 1 has (2 inputs * 16 weights) + 16 biases = 48 parameters to learn.

## Step 3: Compiling the Model

Before the network can learn, we must mathematically define *how* it should learn. This is called compiling the model.

1. **Loss Function:** `binary_crossentropy`. This compares the predicted probability against the true label (0 or 1) and penalizes the network heavily if it is confidently wrong.
2. **Optimizer:** `Adam`. This is the engine of Gradient Descent. It calculates the slope of the loss landscape and updates the weights. Adam is adaptive, meaning it dynamically adjusts the learning rate for each individual parameter during training.
3. **Metrics:** We track `accuracy` to understand the model's performance in human-readable terms.

In [ ]:
# Cell 8: Model Compilation

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

print("Model successfully compiled and ready for training.")

## Step 4: Executing the Training Loop (Fitting)

We will now train the model. 
* **Epochs:** The network will pass over the entire dataset 100 times.
* **Validation Split:** TensorFlow will hold back 20% of the training data during learning to serve as an internal check against overfitting at the end of each epoch.

TensorFlow abstracts the complex calculus of Backpropagation away from us. When we call `.fit()`, the matrix multiplication and weight updates happen automatically on the backend CPU/GPU.

In [ ]:
# Cell 10: Model Training

print("--- Starting Backpropagation ---")

# The history object stores the loss and accuracy metrics for every epoch
history = model.fit(
    X_train, 
    y_train, 
    epochs=100, 
    validation_split=0.2, # Use 20% of training data for mid-training validation
    verbose=0 # Set to 1 or 2 to see the progress bar per epoch
)

print("--- Training Complete ---")

# Let's check the final accuracy on the unseen Test Set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy: {test_accuracy * 100:.2f}%")

## Step 5: Analytical Evaluation and Visualization

A Neural Network is a "Black Box", but we can peek inside by visualizing its outputs. 

First, we will plot the **Learning Curves** to ensure the model didn't overfit (which happens if Validation Loss spikes while Training Loss drops). 
Second, we will map out the continuous 2D feature space to see the actual shape of the mathematical decision boundary the network learned!

In [ ]:
# Cell 12: Visualizing Learning Curves and Decision Boundaries

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Plotting the Learning Curves
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Plotting the Decision Boundary
# Create a dense grid spanning the feature space
x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

# Predict probabilities for every point on the grid
grid_inputs = np.c_[xx.ravel(), yy.ravel()]
predictions = model.predict(grid_inputs, verbose=0)
Z = predictions.reshape(xx.shape)

# Plot the contour map of the probabilities
contour = axes[1].contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.7)
# Overlay the original test data
axes[1].scatter(X_test[y_test==0][:, 0], X_test[y_test==0][:, 1], edgecolors='k', label='Class 0')
axes[1].scatter(X_test[y_test==1][:, 0], X_test[y_test==1][:, 1], edgecolors='k', label='Class 1')

axes[1].set_title('Learned Neural Network Decision Boundary', fontsize=14)
axes[1].set_xlabel('Feature X1')
axes[1].set_ylabel('Feature X2')
axes[1].legend()

plt.tight_layout()
plt.show()

## Conclusion

You have successfully built, trained, and analyzed a Neural Network using TensorFlow!

**Key Analytical Takeaways:**
1. **The Power of Depth:** Standard linear regression would have drawn a useless line through the middle of the circles. Our MLP learned to draw a perfect topological ring around the inner cluster.
2. **Activations are Crucial:** Without the `relu` layers bending the matrix multiplications, the network would have mathematically collapsed into a linear classifier, failing to solve the dataset.
3. **Loss Minimization:** The learning curve proved that Gradient Descent systematically found the bottom of the loss landscape over 100 epochs, improving accuracy with every step.

This exact `Sequential` architecture and training loop is the foundation for almost all advanced Deep Learning, from predicting stock markets to analyzing medical data!